# 🐲 Automated Stereogram 3D Depth Map Generator on Google Colab (NVIDIA A100)

This notebook implements an automated end-to-end pipeline to generate **high-quality volumetric depth maps** for autostereograms:

### 🔄 Pipeline Overview:
1. **Text LLM Prompt Generator (Ollama / OpenRouter / Procedural)**: Synthesizes prompts for 3D figures specifically designed for autostereograms (dramatic forward relief planes, isolated on pure black backgrounds, studio rim lighting, matte clay or octane finish).
2. **Qwen-Image-2.1 Diffusion Model**: Renders the high-fidelity 3D figure with clean silhouette boundaries.
3. **Depth Anything V2 Model**: Hugging Face metric relative depth estimation extracts pure grayscale depth maps ($255 = \text{near}, 0 = \text{far}$).
4. **Looping & Batch Generation**: Generate 5, 20, or 100 depth maps in a single automated run.
5. **Google Drive Persistence**: All paired 3D figures and depth maps are automatically saved to `My Drive/Stereogram_Depthmaps`.
6. **Dual Modes**: Run interactively in the **ComfyUI Web Interface** via Ngrok tunnel, or run headlessly using the **Automated Python Batch Loop**.

## 1. Verify NVIDIA A100 GPU
Ensure you have an A100 GPU connected (`Runtime` -> `Change runtime type` -> select `A100 GPU`).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Outputs
All generated 3D figures and depth maps will be saved directly to your Google Drive (`MyDrive/Stereogram_Depthmaps`).

In [ ]:
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Stereogram_Depthmaps"
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "3D_Figures"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "Stereogram_Depthmaps"), exist_ok=True)
print(f"✓ Google Drive mounted! Output directories ready at:\n  {DRIVE_OUTPUT_DIR}")

## 3. Install ComfyUI & Custom Nodes
Clones ComfyUI, installs updated dependencies, and sets up custom nodes for **Depth Anything V2**, **rgthree**, and **ComfyUI-Manager**.

In [ ]:
import os

%cd /content
if not os.path.exists("/content/ComfyUI"):
    print("Cloning ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    print("ComfyUI already cloned.")

%cd /content/ComfyUI
print("Installing base ComfyUI requirements...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt

print("Installing custom nodes...")
%cd /content/ComfyUI/custom_nodes

custom_nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-DepthAnythingV2", "https://github.com/kijai/ComfyUI-DepthAnythingV2.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-Custom-Scripts", "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),
]

for name, git_url in custom_nodes:
    if not os.path.exists(f"/content/ComfyUI/custom_nodes/{name}"):
        print(f"  -> Cloning {name}...")
        !git clone -q {git_url}
    else:
        print(f"  ✓ {name} already present.")

print("Installing dependencies (huggingface_hub, transformers, diffusers, accelerate)...")
!pip install -q -U torchao "huggingface_hub>=0.28.0" "transformers>=4.45.0" accelerate protobuf sentencepiece timm einops pyngrok requests pillow
!pip install -q git+https://github.com/huggingface/diffusers.git
!apt-get -y update -qq && apt-get -y install -qq lsof aria2 zstd

print("✓ ComfyUI and custom nodes installed successfully!")

## 4. Download Qwen-Image-2.1 & Depth Anything V2 Models (`aria2c`)
Uses multi-connection parallel `aria2c` (16 connections per file) to download weights into their correct ComfyUI directories.

In [ ]:
import os

%cd /content/ComfyUI
os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/depthanything", exist_ok=True)

download_list = [
    # 1. Qwen-Image-2.1 Diffusion Model (BF16 or INT8)
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors",
        "dir": "models/diffusion_models",
        "file": "qwen_image_2.1_bf16.safetensors",
        "desc": "Qwen-Image-2.1 DiT (~14.5 GB)"
    },
    # 2. Qwen3-VL 8B Text Encoder
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_bf16.safetensors",
        "dir": "models/text_encoders",
        "file": "qwen3vl_8b_bf16.safetensors",
        "desc": "Qwen3-VL 8B Text Encoder (~8.5 GB)"
    },
    # 3. Qwen-Image-2.1 VAE
    {
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        "dir": "models/vae",
        "file": "qwen_image_2.1_vae_bf16.safetensors",
        "desc": "Qwen-Image-2.1 VAE (~0.5 GB)"
    },
    # 4. Depth Anything V2 Large Model (Safetensors for Kijai node)
    {
        "url": "https://huggingface.co/Kijai/DepthAnythingV2-safetensors/resolve/main/depth_anything_v2_vitl_fp32.safetensors",
        "dir": "models/depthanything",
        "file": "depth_anything_v2_vitl_fp32.safetensors",
        "desc": "Depth Anything V2 Large (~1.3 GB)"
    }
]

print("Starting parallel model downloads...")
for item in download_list:
    dest_path = os.path.join(item["dir"], item["file"])
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < 1000000:
        print(f"\n📥 Downloading {item['file']} ({item['desc']})...")
        !aria2c -c -x 16 -s 16 -k 1M "{item['url']}" -d "{item['dir']}" -o "{item['file']}"
    else:
        print(f"  ✓ Already downloaded: {item['file']} ({os.path.getsize(dest_path) / (1024**3):.2f} GB)")

print("\n🎉 All model weights downloaded and ready!")

## 5. Install the ComfyUI Stereogram Workflow
Saves `workflow_stereogram_depthmap_generator.json` directly into ComfyUI's menu workflows and copies it to `/content`.

In [ ]:
import json
import os
import shutil

WORKFLOW_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WORKFLOW_DIR, exist_ok=True)
WORKFLOW_NAME = "workflow_stereogram_depthmap_generator.json"
TARGET_PATH = os.path.join(WORKFLOW_DIR, WORKFLOW_NAME)

# Pre-configured graph: Qwen-Image-2.1 -> VAEDecode -> DepthAnything_V2 -> Save to Drive
WORKFLOW_DATA = {
  "last_node_id": 12,
  "last_link_id": 12,
  "nodes": [
    {
      "id": 1,
      "type": "UNETLoader",
      "pos": [40, 80],
      "size": [320, 82],
      "widgets_values": ["qwen_image_2.1_bf16.safetensors"],
      "outputs": [{"name": "MODEL", "type": "MODEL", "links": [1], "slot_index": 0}]
    },
    {
      "id": 2,
      "type": "CLIPLoader",
      "pos": [40, 200],
      "size": [320, 106],
      "widgets_values": ["qwen3vl_8b_bf16.safetensors", "default"],
      "outputs": [{"name": "CLIP", "type": "CLIP", "links": [2, 3], "slot_index": 0}]
    },
    {
      "id": 3,
      "type": "VAELoader",
      "pos": [40, 350],
      "size": [320, 82],
      "widgets_values": ["qwen_image_2.1_vae_bf16.safetensors"],
      "outputs": [{"name": "VAE", "type": "VAE", "links": [4], "slot_index": 0}]
    },
    {
      "id": 4,
      "type": "EmptyLatentImage",
      "pos": [40, 480],
      "size": [320, 106],
      "widgets_values": [1024, 1024, 1],
      "outputs": [{"name": "LATENT", "type": "LATENT", "links": [5], "slot_index": 0}]
    },
    {
      "id": 5,
      "type": "CLIPTextEncode",
      "pos": [420, 80],
      "size": [440, 180],
      "title": "Positive Prompt (3D Stereogram Figure)",
      "inputs": [{"name": "clip", "type": "CLIP", "link": 2}],
      "widgets_values": ["A highly detailed 3D digital sculpture of an ancient dragon perched on a rock, wings spread forward, dramatic volumetric relief with sharp protruding horns and snout, matte clay finish, studio rim lighting, pure solid black background, stereogram 3D depth figure, octane render, 8k"],
      "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6], "slot_index": 0}]
    },
    {
      "id": 6,
      "type": "CLIPTextEncode",
      "pos": [420, 300],
      "size": [440, 120],
      "title": "Negative Prompt",
      "inputs": [{"name": "clip", "type": "CLIP", "link": 3}],
      "widgets_values": ["blurry, low quality, flat, noisy, cluttered background, multiple subjects, text, watermark, cropped, complex noisy background"],
      "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7], "slot_index": 0}]
    },
    {
      "id": 7,
      "type": "KSampler",
      "pos": [900, 80],
      "size": [320, 474],
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 1},
        {"name": "positive", "type": "CONDITIONING", "link": 6},
        {"name": "negative", "type": "CONDITIONING", "link": 7},
        {"name": "latent_image", "type": "LATENT", "link": 5}
      ],
      "widgets_values": [42, "randomize", 30, 4.0, "euler", "simple", 1.0],
      "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8], "slot_index": 0}]
    },
    {
      "id": 8,
      "type": "VAEDecode",
      "pos": [1260, 80],
      "size": [210, 46],
      "inputs": [{"name": "samples", "type": "LATENT", "link": 8}, {"name": "vae", "type": "VAE", "link": 4}],
      "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [9, 10], "slot_index": 0}]
    },
    {
      "id": 9,
      "type": "SaveImage",
      "pos": [1510, 80],
      "size": [350, 380],
      "title": "Save 3D Figure (RGB Render)",
      "inputs": [{"name": "images", "type": "IMAGE", "link": 9}],
      "widgets_values": ["3D_Figures/figure"]
    },
    {
      "id": 10,
      "type": "DownloadAndLoadDepthAnythingV2Model",
      "pos": [1260, 480],
      "size": [340, 82],
      "title": "Depth Anything V2 Loader",
      "widgets_values": ["depth_anything_v2_vitl_fp32.safetensors"],
      "outputs": [{"name": "da_v2_model", "type": "DAMODEL", "links": [11], "slot_index": 0}]
    },
    {
      "id": 11,
      "type": "DepthAnything_V2",
      "pos": [1640, 480],
      "size": [280, 80],
      "title": "Compute Grayscale Depth Map",
      "inputs": [{"name": "da_model", "type": "DAMODEL", "link": 11}, {"name": "images", "type": "IMAGE", "link": 10}],
      "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [12], "slot_index": 0}]
    },
    {
      "id": 12,
      "type": "SaveImage",
      "pos": [1960, 480],
      "size": [360, 380],
      "title": "Save Grayscale Depth Map (Stereogram)",
      "inputs": [{"name": "images", "type": "IMAGE", "link": 12}],
      "widgets_values": ["Stereogram_Depthmaps/depthmap"]
    }
  ],
  "links": [
    [1, 1, 0, 7, 0, "MODEL"],
    [2, 2, 0, 5, 0, "CLIP"],
    [3, 2, 0, 6, 0, "CLIP"],
    [4, 3, 0, 8, 1, "VAE"],
    [5, 4, 0, 7, 3, "LATENT"],
    [6, 5, 0, 7, 1, "CONDITIONING"],
    [7, 6, 0, 7, 2, "CONDITIONING"],
    [8, 7, 0, 8, 0, "LATENT"],
    [9, 8, 0, 9, 0, "IMAGE"],
    [10, 8, 0, 11, 1, "IMAGE"],
    [11, 10, 0, 11, 0, "DAMODEL"],
    [12, 11, 0, 12, 0, "IMAGE"]
  ],
  "version": 0.4
}

with open(TARGET_PATH, "w", encoding="utf-8") as f:
    json.dump(WORKFLOW_DATA, f, indent=2)

shutil.copy(TARGET_PATH, f"/content/{WORKFLOW_NAME}")
print(f"✓ Workflow installed in ComfyUI menu: {TARGET_PATH}")
print(f"✓ Drag-and-drop workflow file available at: /content/{WORKFLOW_NAME}")

## 6. (Optional) Run Free Local Text LLM (Ollama) in Colab
Runs a fast, local LLM (`qwen2.5:7b` or `minicpm-v`) inside Colab on port 11434 to automatically craft creative 3D stereogram figure prompts.

In [ ]:
import subprocess
import time
import urllib.request
import os

print("Installing Ollama daemon...")
!curl -fsSL https://ollama.com/install.sh | sh

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.run("pkill -9 ollama 2>/dev/null", shell=True)
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(2)

for _ in range(15):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1) as resp:
            if resp.status == 200:
                print("✓ Ollama is running and healthy on port 11434!")
                break
    except Exception:
        time.sleep(1)

print("Pulling Qwen2.5 7B text model for stereogram prompt generation...")
!ollama pull qwen2.5:7b
print("✓ Local prompt LLM ready!")

## 7. Launch ComfyUI + Dual Tunnel (Ngrok & localhost.run)
Starts ComfyUI and creates a direct public URL to access the web interface.

Paste your **Ngrok Auth Token** below (or leave empty to use the `localhost.run` fallback tunnel).

In [ ]:
import subprocess
import threading
import time
import os
import re

PORT = 8199
LOG_FILE = '/tmp/comfyui.log'
FORCE_LOCALHOST_RUN = False
OUTPUT_DIR = globals().get('DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/Stereogram_Depthmaps')

# Clean up previous processes and ports
subprocess.run('pkill -9 -f ngrok 2>/dev/null', shell=True)
subprocess.run(f'fuser -k -9 {PORT}/tcp 8188/tcp 2>/dev/null || kill -9 $(lsof -n -P -t -i:{PORT},8188 2>/dev/null) 2>/dev/null', shell=True)

# Start ComfyUI daemon
cmd = [
    'python', 'main.py',
    '--listen', '0.0.0.0',
    '--port', str(PORT),
    '--output-directory', OUTPUT_DIR,
    '--highvram'
]

comfyui = subprocess.Popen(cmd, cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

def stream_to_file():
    with open(LOG_FILE, 'w') as log:
        for line in comfyui.stdout:
            log.write(line.decode('utf-8', errors='replace'))
            log.flush()

threading.Thread(target=stream_to_file, daemon=True).start()
print(f"ComfyUI starting... logs streaming to: {LOG_FILE}")
time.sleep(5)

# ── Tunnel Setup (Ngrok with localhost.run fallback) ──
NGROK_TOKEN = ""  # Optional: Paste your ngrok token here
url = None

if not FORCE_LOCALHOST_RUN and NGROK_TOKEN.strip():
    try:
        from pyngrok import ngrok, conf
        ngrok.kill()
        conf.get_default().auth_token = NGROK_TOKEN.strip()
        ngrok.set_auth_token(NGROK_TOKEN.strip())
        tunnel_obj = ngrok.connect(PORT, "http")
        url = tunnel_obj.public_url
        if url:
            print('✓ Tunnel established: ngrok')
    except Exception as e:
        print(f'ngrok connection failed ({e}), falling back to localhost.run...')
        url = None

if not url:
    tunnel = subprocess.Popen(
        ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'ServerAliveInterval=60',
         '-R', f'80:localhost:{PORT}', 'localhost.run'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print('Connecting via localhost.run...')
    deadline = time.time() + 45
    while time.time() < deadline:
        line = tunnel.stdout.readline().decode('utf-8', errors='replace')
        match = re.search(r'https?://(?!admin\.)[a-zA-Z0-9\-]+\.lhr\.life', line)
        if match:
            url = match.group(0)
            print('✓ Tunnel established: localhost.run')
            break

print()
print('=' * 65)
print('  🚀 COMFYUI WEB INTERFACE URL:')
print('    ', url or '(tunnel connecting... check logs)')
print('=' * 65)
print(f'  📁 GOOGLE DRIVE OUTPUTS: {OUTPUT_DIR}')
print(f'  📄 COMFYUI LOGS:         tail -f {LOG_FILE}')
print("\n👉 How to loop in ComfyUI: Set 'Batch count' to 10 in the Queue options!")

## 8. Automated Batch Loop (ComfyUI API / Standalone)
Loops over `NUM_IMAGES` to generate stereogram 3D figures and Depth Anything V2 maps automatically.

**Features:**
- **Direct ComfyUI Connection**: If ComfyUI is running from Cell 7, sends jobs directly to ComfyUI's local API (fastest, uses already downloaded models in VRAM).
- **Standalone Fallback**: If ComfyUI is not running, falls back to direct pipeline execution with auto-resolved dependencies.

In [ ]:
import os
import sys
import time
import json
import random
import glob
import urllib.request
import urllib.parse
from PIL import Image
from IPython.display import display

# ================== BATCH CONFIGURATION ==================
NUM_IMAGES = 5            # Number of stereogram depth maps to create
RESOLUTION_SCALE = 0.5    # 0.5 -> 1024x1024 (fast 1K, recommended), 1.0 -> 2048x2048
USE_OLLAMA_PROMPTS = True # True: queries Ollama on port 11434; False: procedural synthesizer
COMFY_PORT = 8199         # ComfyUI port from Cell 7
# =========================================================

DRIVE_DIR = globals().get('DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/Stereogram_Depthmaps')
FIG_DIR = os.path.join(DRIVE_DIR, '3D_Figures')
DEPTH_DIR = os.path.join(DRIVE_DIR, 'Stereogram_Depthmaps')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DEPTH_DIR, exist_ok=True)

# ── Detect ComfyUI API on localhost ──
comfy_url = None
for port in [COMFY_PORT, 8188]:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/system_stats", timeout=1.5) as resp:
            if resp.status == 200:
                comfy_url = f"http://127.0.0.1:{port}"
                break
    except Exception:
        pass

# ── Prompt Generation Helper ──
SUBJECTS = [
    "an ancient mythological dragon perched on a rock with wings swept forward",
    "a roaring sabertooth tiger in dynamic predatory stance",
    "a coiled king cobra with wide spread hood facing forward",
    "a majestic rearing Pegasus with outstretched feathered wings",
    "an ornate gothic gargoyle with folded wings and protruding horns",
    "a biomechanical armored scarab beetle with metallic ridges",
    "a deep-sea giant kraken with tentacles reaching toward the viewer",
    "a cybernetic skull with horned crown and glowing mechanical eyes",
    "a predatory Tyrannosaurus Rex with wide open jaws",
    "a fantasy phoenix rising with fiery forward-swept wings"
]
MATERIALS = ["matte gray clay", "chiseled Carrara marble", "dark oxidized bronze", "matte ceramic porcelain", "fine-grain volcanic basalt stone"]

def get_stereogram_prompt():
    if USE_OLLAMA_PROMPTS:
        try:
            req = urllib.request.Request(
                "http://127.0.0.1:11434/api/generate",
                data=json.dumps({
                    "model": "qwen2.5:7b",
                    "prompt": "Write one single prompt for an image of a 3D figure with dramatic forward-protruding depth (snout, horns, or limbs), matte clay or stone, isolated on solid black background, octane render, clean silhouette, 8k. Output only the prompt string.",
                    "stream": False
                }).encode('utf-8'),
                headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=12) as r:
                res = json.loads(r.read().decode('utf-8')).get('response', '').strip()
                if len(res) > 20:
                    return res.replace('"', '').replace('**', '').replace('Prompt:', '').strip()
        except Exception:
            pass
    sub = random.choice(SUBJECTS)
    mat = random.choice(MATERIALS)
    return f"A 3D digital sculpture of {sub}, volumetric relief with protruding features, {mat} finish, studio rim lighting, isolated on solid pure black background, octane render, 8k"

dim = int(round((2048 * RESOLUTION_SCALE) / 32)) * 32

if comfy_url:
    print(f"🚀 Executing Batch Loop via ComfyUI API ({comfy_url})...")
    print(f"   Output Directory: {DRIVE_DIR}")
    print(f"   Batch Count:      {NUM_IMAGES}\n")
    
    def queue_comfy(prompt_text, seed):
        wf = {
            "1": {"inputs": {"unet_name": "qwen_image_2.1_bf16.safetensors"}, "class_type": "UNETLoader"},
            "2": {"inputs": {"clip_name": "qwen3vl_8b_bf16.safetensors", "type": "default"}, "class_type": "CLIPLoader"},
            "3": {"inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}, "class_type": "VAELoader"},
            "4": {"inputs": {"width": dim, "height": dim, "batch_size": 1}, "class_type": "EmptyLatentImage"},
            "5": {"inputs": {"text": prompt_text, "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
            "6": {"inputs": {"text": "blurry, low quality, flat, noisy, cluttered background, multiple subjects, text, watermark, cropped", "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
            "7": {"inputs": {"seed": seed, "steps": 30, "cfg": 4.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0, "model": ["1", 0], "positive": ["5", 0], "negative": ["6", 0], "latent_image": ["4", 0]}, "class_type": "KSampler"},
            "8": {"inputs": {"samples": ["7", 0], "vae": ["3", 0]}, "class_type": "VAEDecode"},
            "9": {"inputs": {"filename_prefix": "3D_Figures/figure", "images": ["8", 0]}, "class_type": "SaveImage"},
            "10": {"inputs": {"model": "depth_anything_v2_vitl_fp32.safetensors"}, "class_type": "DownloadAndLoadDepthAnythingV2Model"},
            "11": {"inputs": {"da_model": ["10", 0], "images": ["8", 0]}, "class_type": "DepthAnything_V2"},
            "12": {"inputs": {"filename_prefix": "Stereogram_Depthmaps/depthmap", "images": ["11", 0]}, "class_type": "SaveImage"}
        }
        data = json.dumps({"prompt": wf, "client_id": "colab_runner"}).encode('utf-8')
        req = urllib.request.Request(f"{comfy_url}/prompt", data=data, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode('utf-8')).get("prompt_id")
    
    for idx in range(1, NUM_IMAGES + 1):
        prompt = get_stereogram_prompt()
        seed = random.randint(1, 2147483647)
        print(f"[{idx}/{NUM_IMAGES}] Queuing to ComfyUI (Seed: {seed}):")
        print(f"   Prompt: \"{prompt}\"")
        
        pid = queue_comfy(prompt, seed)
        print(f"   -> Processing job {pid}...")
        
        # Wait for job completion
        done = False
        for _ in range(150):
            time.sleep(2)
            try:
                with urllib.request.urlopen(f"{comfy_url}/history/{pid}") as resp:
                    h = json.loads(resp.read().decode('utf-8'))
                    if pid in h:
                        done = True
                        break
            except Exception:
                pass
        
        if done:
            print(f"   ✓ Generation complete!")
            # Show preview of latest depthmap
            depth_files = sorted(glob.glob(os.path.join(DEPTH_DIR, "*.*epth*")), key=os.path.getmtime)
            if depth_files:
                display(Image.open(depth_files[-1]))
        else:
            print("   ⚠️ Job still running in background, continuing loop...")

else:
    print("ℹ️ ComfyUI daemon not found on port 8199. Using direct diffusers pipeline...")
    import subprocess
    # Ensure required dependencies are installed without version conflicts
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.28.0", "transformers>=4.45.0", "accelerate"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/diffusers.git"], check=True)
    
    for mod in list(sys.modules.keys()):
        if mod.startswith("diffusers") or mod.startswith("huggingface_hub"):
            del sys.modules[mod]
            
    from diffusers import QwenImage21Pipeline
    from transformers import pipeline as hf_pipeline
    import torch
    import numpy as np
    
    qwen_pipe = QwenImage21Pipeline.from_pretrained("Qwen/Qwen-Image-2.1", torch_dtype=torch.bfloat16)
    if torch.cuda.get_device_properties(0).total_memory / (1024**3) > 50:
        qwen_pipe.to("cuda")
    else:
        qwen_pipe.enable_model_cpu_offload()
        
    depth_pipe = hf_pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Large-hf", device=0, torch_dtype=torch.bfloat16)
    
    for idx in range(1, NUM_IMAGES + 1):
        prompt = get_stereogram_prompt()
        seed = random.randint(1, 2147483647)
        print(f"[{idx}/{NUM_IMAGES}] Prompt: {prompt} (Seed: {seed})")
        res = qwen_pipe(prompt=prompt, width=dim, height=dim, num_inference_steps=30, true_cfg_scale=4.0, generator=torch.Generator(device="cuda").manual_seed(seed))
        rgb = res.images[0]
        rgb.save(os.path.join(FIG_DIR, f"figure_{idx:03d}_seed{seed}.png"))
        
        depth_res = depth_pipe(rgb)
        depth_raw = np.array(depth_res["depth"]).astype(np.float32)
        d_min, d_max = depth_raw.min(), depth_raw.max()
        depth_norm = (((depth_raw - d_min) / (d_max - d_min + 1e-6)) * 255.0).astype(np.uint8)
        depth_img = Image.fromarray(depth_norm, mode="L")
        depth_img.save(os.path.join(DEPTH_DIR, f"depthmap_{idx:03d}_seed{seed}.png"))
        display(depth_img)

print(f"\n🎉 Batch generation complete! All assets saved to {DRIVE_DIR}")